# 11 — Diverse Rank Ensemble (anchored)
Rank-average three submissions with anchored weights: 60% to the verified 95.28% ECOD-plus-temporal baseline, 20% each to autoencoder and isolation-forest.

**Rationale.** Spearman vs ECOD-plus-temporal: AE = 0.79, IF = 0.79 (room to diversify). CTST = -0.31 (excluded — anti-correlated, would hurt). The 60% ECOD anchor is downside protection: an unweighted rank-avg of these three has the same 4-model failure mode the user already hit at 92%.

Writes `submission_diverse.csv`.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from scipy.stats import rankdata, spearmanr

out = ROOT / 'outputs' / 'submissions'
ecod = pd.read_csv(out / 'submission_ecod_plus_temporal.csv').sort_values('ID').reset_index(drop=True)
ae = pd.read_csv(out / 'submission_autoencoder.csv').sort_values('ID').reset_index(drop=True)
iforest = pd.read_csv(out / 'submission_isolation_forest.csv').sort_values('ID').reset_index(drop=True)
assert (ecod['ID'].to_numpy() == ae['ID'].to_numpy()).all()
assert (ecod['ID'].to_numpy() == iforest['ID'].to_numpy()).all()
n = len(ecod)
print('rows:', n)

rows: 289101


In [2]:
r_ecod = rankdata(ecod['TARGET'].to_numpy()) / n
r_ae = rankdata(ae['TARGET'].to_numpy()) / n
r_if = rankdata(iforest['TARGET'].to_numpy()) / n

score = 0.60 * r_ecod + 0.20 * r_ae + 0.20 * r_if
print('ensemble vs ecod corr:', round(float(spearmanr(score, r_ecod)[0]), 4))
print('min/max/mean:', round(float(score.min()), 4), round(float(score.max()), 4), round(float(score.mean()), 4))

pd.DataFrame({'ID': ecod['ID'], 'TARGET': score}).to_csv(out / 'submission_diverse.csv', index=False)

ensemble vs ecod corr: 0.9708
min/max/mean: 0.003 0.9977 0.5
